# 07. Descriptive statistics (Table S1, Figures 3 and 4)

Differences between Democrats (`partyid_1 == 1`) and Republicans (`partyid_1 == 2`) on the variables of the final model.

- **Table S1**: weighted mean (SE) for continuous variables (`type == con`, weighted t-test) and weighted percentages for
  categorical variables (`type == nom`, chi-square test), for the 66 variables of the final model (sheet `XGBVARS` of
  `inputs/top_40_features_m3.xlsx`). Statistics for each variable use the respondents in the ballot-years in which the
  variable entered the model.
- **Figures 3 and 4**: weighted mean responses of Democrats and Republicans by year with 95% confidence intervals, for the
  six attitude variables of the reduced model in Figure 2-F (Figure 3) and three SES variables (Figure 4).

Weights: `wtssnrps` from 2004 on (when valid), `wtssps` otherwise (missing/0 → 1).

**Input**: `data/data_for_final_analysis_M3_Top40.pkl` (from `05_final_model.ipynb`), `inputs/top_40_features_m3.xlsx`

**Output** (`outputs/4_descriptive/`): `4_Ttest_descriptive_table_XGB.{tex,xlsx}` (Table S1), `trend_plots/trend_con_*.png` (Figures 3 and 4)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as sps
import statsmodels.stats.weightstats as sm_weightstats
import warnings
import os

warnings.filterwarnings('ignore')

# --- File paths ---
PICKLE_DATA_FILE = "data/data_for_final_analysis_M3_Top40.pkl"
FEATURE_LIST_FILE = "inputs/top_40_features_m3.xlsx"


def get_significance_stars(p_val):
    """p-value -> significance stars."""
    if pd.isna(p_val):
        return ''
    if p_val < 0.001:
        return '***'
    elif p_val < 0.01:
        return '**'
    elif p_val < 0.05:
        return '*'
    return ''


def coerce_numeric_safe(series):
    """Replacement for pd.to_numeric(errors='ignore') (deprecated in pandas 2.2).
    Convert if every value can be converted; otherwise return the original series."""
    converted = pd.to_numeric(series, errors='coerce')
    if converted.notna().sum() == series.notna().sum():
        return converted
    return series


def load_data_from_pickle(pickle_file=PICKLE_DATA_FILE):
    """Load the model data {(ballot, model): {year: df}}."""
    print(f"--- Loading data from '{pickle_file}'... ---")
    if not os.path.exists(pickle_file):
        print(f"!! Fatal error: '{pickle_file}' not found.")
        return None
    try:
        data = pd.read_pickle(pickle_file)
        if not isinstance(data, dict):
            print(f"!! Error: '{pickle_file}' is not a dict.")
            return None
        print("--- Data loaded. ---")
        return data
    except Exception as e:
        print(f"!! Error while loading the pickle: {e}")
        return None


def create_final_weight(df):
    """Add final_weight (year >= 2004 and valid wtssnrps -> wtssnrps, otherwise wtssps)."""
    if 'wtssnrps' in df.columns and 'wtssps' in df.columns:
        df['final_weight'] = np.where(
            (pd.to_numeric(df['year'], errors='coerce') >= 2004) & (df['wtssnrps'].notna()) & (df['wtssnrps'] > 0),
            df['wtssnrps'],
            df['wtssps']
        )
    elif 'wtssps' in df.columns:
        df['final_weight'] = df['wtssps']
    else:
        df['final_weight'] = 1.0
    df['final_weight'] = df['final_weight'].fillna(1.0).replace(0, 1.0)
    return df


def build_master_dfs(data_for_analysis, models=('XGBoost',)):
    """{(ballot, model): {year: df}} -> {model_name: master_df}.
    Adds 'year' and 'final_weight' to each master_df; None for models without data."""
    print("--- Combining years and ballots... ---")
    if not data_for_analysis:
        print("!! Error: empty input data.")
        return {m: None for m in models}
    buckets = {m: [] for m in models}
    for (ballot, model_name), year_dict in data_for_analysis.items():
        if model_name not in buckets:
            continue
        for year, df in year_dict.items():
            if df is not None and not df.empty:
                df_copy = df.copy()
                df_copy['year'] = year
                buckets[model_name].append(df_copy)
    masters = {}
    for m in models:
        if buckets[m]:
            mdf = pd.concat(buckets[m], ignore_index=True)
            masters[m] = create_final_weight(mdf)
            print(f"  - {m} master data: {len(masters[m])} rows")
        else:
            masters[m] = None
            print(f"  - {m}: no data -> None")
    return masters


def build_feature_frame(data_for_analysis, feature, model='XGBoost'):
    """Rows of the (ballot, year) cells in which `feature` entered the model: [year, partyid_1, final_weight, feature].
    The model notebooks only include a variable in a cell if it is observed there, so the rows of cells containing
    the column define where the variable was used, and NaN within them are the missing values the model received.
    (Pooling all ballots/years would count respondents of cells where the item was not asked as missing.)"""
    frames = []
    for (ballot, model_name), year_dict in data_for_analysis.items():
        if model_name != model:
            continue
        for year, df in year_dict.items():
            if df is None or df.empty or feature not in df.columns:
                continue
            sub = df[[c for c in ['partyid_1', 'wtssnrps', 'wtssps', feature] if c in df.columns]].copy()
            sub[feature] = coerce_numeric_safe(sub[feature].astype(object))
            sub['year'] = year
            frames.append(sub)
    if not frames:
        return None
    return create_final_weight(pd.concat(frames, ignore_index=True))


def weighted_missing_by_party(frame, feature):
    """Weighted (missing N, total N) by partyid_1 (1/2) -> {party: (n_missing, n_total)}."""
    out = {}
    for party in (1, 2):
        g = frame[frame['partyid_1'] == party]
        out[party] = (g.loc[g[feature].isna(), 'final_weight'].sum(), g['final_weight'].sum())
    return out


print("[Setup] Helpers defined.")

## Table S1: descriptive statistics of the final model

In [ ]:
# === Table S1: descriptive statistics (LaTeX longtable + Excel) ===
# Variable metadata (label, measure, type, category) come from sheet XGBVARS of top_40_features_m3.xlsx.
import re

CATEGORY_ORDER = ['Demography', 'SES', 'Government', 'Immigrant', 'Race',
                  'Religion', 'Sex/Family', 'Social Capital', 'Lifestyle']
MODEL_SHEET = {'XGBoost': 'XGBVARS'}
MODEL_TEX = {'XGBoost': 'outputs/4_descriptive/4_Ttest_descriptive_table_XGB.tex'}
MODEL_TAG = {'XGBoost': 'xgb'}
MODEL_XLSX = {'XGBoost': 'outputs/4_descriptive/4_Ttest_descriptive_table_XGB.xlsx'}
os.makedirs('outputs/4_descriptive', exist_ok=True)

# Rows of the Excel version of the table (filled in the variable loop)
TABLE_RECORDS = []

# Response options added/corrected where the xlsx is empty or does not match the recodes
MEASURE_OVERRIDE = {
    # 2 and 3 swapped in data preparation (original GSS: 1 better; 2 worse; 3 stayed same)
    'finalter': '1 better; 2 stayed same; 3 worse',
    # 6 (no children) set to missing in data preparation
    'kidssol_c': '1 much better; 2 somewhat better; 3 about the same; 4 somewhat worse; 5 much worse',
    # 5 (no class) set to missing in data preparation
    'class_c': '1 lower class; 2 working class; 3 middle class; 4 upper class',
    # no response options in the xlsx (GSS codebook)
    'degree': "0 less than high school; 1 high school; 2 associate/junior college; 3 bachelor's; 4 graduate",
    # only 1/5/7 in the xlsx (GSS codebook)
    'incom16': '1 far below average; 2 below average; 3 average; 4 above average; 5 far above average; 7 lived in institution',
}


def add_record(category, var, label, row, dem_a, dem_b, rep_a, rep_b, p=None):
    """Record one row of the Excel table. a = mean or weighted N, b = SE or %."""
    TABLE_RECORDS.append({'Category': category, 'Variable': var, 'Label': label, 'Row': row,
                          'Democrat': dem_a, 'Democrat (SE or %)': dem_b,
                          'Republican': rep_a, 'Republican (SE or %)': rep_b, 'p-value': p})


def latex_escape(s):
    s = str(s)
    # Escape '$' first (the '<'/'>' replacements below insert '$' on purpose)
    for a, b in [('$', r'\$'), ('&', r'\&'), ('%', r'\%'), ('#', r'\#'), ('_', r'\_')]:
        s = s.replace(a, b)
    return s.replace('<', r'$<$').replace('>', r'$>$')


def fmt_p(p):
    if pd.isna(p):
        return ''
    p = float(p)
    if p < 0.001:
        return r'$< .001$'
    s = f'{p:.2f}'
    return f'{p:.3f}' if s == '0.00' else s


def parse_measure_string(s):
    d, s = {}, str(s)
    if s.strip() in ('0', '0.0', '0.00', 'nan', ''):
        return d
    for part in s.split(';'):
        part = part.strip()
        if not part:
            continue
        m = re.match(r'(\d+)\s*(.*)', part)
        if m:
            d[m.group(1)] = m.group(2).strip()
        else:
            d[part] = ''
    return d


def _valid_measure(m):
    return str(m).strip() not in ('0', '0.0', '0.00', 'nan', '')


def missing_row(df, feat, category='', label=''):
    """Missing % row. Denominator = weighted N by party in the cells where the variable entered the model."""
    m = weighted_missing_by_party(df, feat)
    (md, td), (mr, tr) = m[1], m[2]
    add_record(category, feat, label, 'Missing', md, (md / td * 100 if td else 0), mr, (mr / tr * 100 if tr else 0))
    return rf'\quad Missing & {(md / td * 100 if td else 0):.1f}\% & {(mr / tr * 100 if tr else 0):.1f}\% & \\'


def con_rows(df, feat, label, measure, category=''):
    sub = df[[feat, 'partyid_1', 'final_weight']].copy()
    sub[feat] = coerce_numeric_safe(sub[feat])
    sub = sub.dropna()
    dem, rep = sub[sub['partyid_1'] == 1], sub[sub['partyid_1'] == 2]
    if dem.empty or rep.empty:
        return None
    d = sm_weightstats.DescrStatsW(dem[feat], weights=dem['final_weight'])
    r = sm_weightstats.DescrStatsW(rep[feat], weights=rep['final_weight'])
    _, p, _ = sm_weightstats.ttest_ind(dem[feat], rep[feat],
                                       weights=(dem['final_weight'], rep['final_weight']), usevar='unequal')
    out = [rf'{latex_escape(label)} & {d.mean:.2f} ({d.std_mean:.3f}) & {r.mean:.2f} ({r.std_mean:.3f}) & {fmt_p(p)} \\']
    add_record(category, feat, label, 'Mean (SE)', d.mean, d.std_mean, r.mean, r.std_mean, p)
    if _valid_measure(measure):
        out.append(rf'\quad \textit{{({latex_escape(str(measure).strip())})}} & & & \\')
    out.append(missing_row(df, feat, category, label))
    return out


def nom_rows(df, feat, label, measure, category=''):
    sub = df[[feat, 'partyid_1', 'final_weight']].copy()
    sub[feat] = coerce_numeric_safe(sub[feat])
    sub = sub.dropna()
    if sub.empty or sub[feat].nunique() < 2:
        return None
    ct = pd.crosstab(sub[feat], sub['partyid_1'], values=sub['final_weight'], aggfunc='sum').fillna(0)
    # Denominator = weighted N by party including missing values (category rows + Missing row = 100%)
    totals = df.groupby('partyid_1')['final_weight'].sum()
    prop = ct.div(totals.reindex(ct.columns), axis=1)
    ctf = ct.loc[ct.sum(axis=1) > 0, ct.sum(axis=0) > 0]
    if ctf.shape[0] < 2 or ctf.shape[1] < 2:
        return None
    _, p, _, _ = sps.chi2_contingency(ctf)
    desc = parse_measure_string(measure)
    out = [rf'{latex_escape(label)} & & & {fmt_p(p)} \\']
    add_record(category, feat, label, '', None, None, None, None, p)
    for cat in ct.index:
        try:
            key = str(int(float(cat)))
        except (ValueError, TypeError):
            key = str(cat)
        nd = ct.loc[cat, 1] if 1 in ct.columns else 0
        pdv = prop.loc[cat, 1] if 1 in prop.columns else 0
        nr = ct.loc[cat, 2] if 2 in ct.columns else 0
        prv = prop.loc[cat, 2] if 2 in prop.columns else 0
        lab = desc.get(key, '')
        lab = (lab[:1].upper() + lab[1:]) if lab else ''
        out.append(rf'\quad {latex_escape((key + " " + lab).strip())} & {pdv * 100:.1f}\% & {prv * 100:.1f}\% & \\')
        add_record(category, feat, label, (key + " " + lab).strip(), nd, pdv * 100, nr, prv * 100)
    out.append(missing_row(df, feat, category, label))
    return out


XLSX_CATEGORY_ORDER = ['', 'Attitude', 'Demography', 'SES', 'Social Capital', 'Lifestyle']


def _fmt_cell(row_name, a, b):
    """Display format as in the tex file: 'Mean (SE)' for continuous variables, otherwise weighted '%'."""
    if pd.isna(a):
        return ''
    if row_name == 'Mean (SE)':
        return f'{a:.2f} ({b:.3f})'
    return f'{b:.1f}%'


def _fmt_p_plain(p):
    if pd.isna(p):
        return ''
    return '< .001' if p < 0.001 else (f'{p:.3f}' if f'{p:.2f}' == '0.00' else f'{p:.2f}')


def write_table_xlsx(records, filename, measures=None):
    """records -> Excel with two sheets: table (formatted) and numeric (raw numbers), ordered by XLSX_CATEGORY_ORDER."""
    num = pd.DataFrame(records)
    order = {c: i for i, c in enumerate(XLSX_CATEGORY_ORDER)}
    num = num.iloc[num['Category'].map(lambda c: order.get(c, 99)).argsort(kind='stable')].reset_index(drop=True)
    table = pd.DataFrame({
        'Category': num['Category'], 'Variable': num['Variable'], 'Label': num['Label'], 'Row': num['Row'],
        'Democrat': [_fmt_cell(r, a, b) for r, a, b in zip(num['Row'], num['Democrat'], num['Democrat (SE or %)'])],
        'Republican': [_fmt_cell(r, a, b) for r, a, b in zip(num['Row'], num['Republican'], num['Republican (SE or %)'])],
        'p-value': num['p-value'].map(_fmt_p_plain),
    })
    # Response options shown in the 'Mean (SE)' row of continuous variables (same as the italic line in the tex file)
    if measures:
        table['Response options'] = [measures.get(v, '') if r == 'Mean (SE)' else ''
                                     for v, r in zip(table['Variable'], table['Row'])]
    with pd.ExcelWriter(filename) as writer:
        table.to_excel(writer, sheet_name='table', index=False)
        num.to_excel(writer, sheet_name='numeric', index=False)


def party_block(df):
    sub = df[df['partyid_1'].isin([1, 2])]
    wd = sub[sub['partyid_1'] == 1]['final_weight'].sum()
    wr = sub[sub['partyid_1'] == 2]['final_weight'].sum()
    tot = wd + wr if (wd + wr) else 1
    add_record('', 'partyid_1', 'Party Identity', '1 Democrat', wd, wd / tot * 100, None, None)
    add_record('', 'partyid_1', 'Party Identity', '2 Republican', None, None, wr, wr / tot * 100)
    return [r'Party Identity & & & \\',
            rf'\quad 1 Democrat & {wd / tot * 100:.1f}\% & & \\',
            rf'\quad 2 Republican & & {wr / tot * 100:.1f}\% & \\']


def preamble(model, n_dem, n_rep):
    """n_dem, n_rep: unweighted numbers of respondents (shown in the caption)"""
    tag = MODEL_TAG[model]
    return (
        r'%\scriptsize' + '\n'
        r'\begin{longtable}{l c c c}' + '\n\n'
        r'\caption{\textbf{Descriptive Statistics of the Final Model (' + model + r')} '
        r'Mean (SE) for continuous variables (SE to three decimals) and weighted percentages for categorical variables. '
        rf'N = {n_dem + n_rep:,} respondents ({n_dem:,} Democrats, {n_rep:,} Republicans; unweighted). '
        r'Missing: weighted percentage of respondents without a valid response among those in the ballot-years in which the variable entered the model; '
        r'for categorical variables, percentages include missing responses in the denominator.} '
        r'\label{tab:demographics_' + tag + r'} \\' + '\n\n'
        r'\toprule' + '\n'
        r'\textbf{Variable} & \textbf{Democrat} & \textbf{Republican} & \textbf{p-value} \\' + '\n'
        r'\midrule' + '\n'
        r'\endfirsthead' + '\n\n'
        r'\multicolumn{4}{c}{{\bfseries \tablename\ \thetable{} -- continued from previous page}} \\' + '\n'
        r'\toprule' + '\n'
        r'\textbf{Variable} & \textbf{Democrat} & \textbf{Republican} & \textbf{p-value} \\' + '\n'
        r'\midrule' + '\n'
        r'\endhead' + '\n\n'
        r'\midrule' + '\n'
        r'\multicolumn{4}{r}{{Continued on next page}} \\' + '\n'
        r'\endfoot' + '\n\n'
        r'\bottomrule' + '\n'
        r'\endlastfoot' + '\n\n')


print("\n" + "=" * 60)
print("Table S1: descriptive statistics")
print("=" * 60)

data_for_analysis = load_data_from_pickle(PICKLE_DATA_FILE)
masters = build_master_dfs(data_for_analysis)

# The final model uses XGBoost only
for model in ['XGBoost']:
    df = masters.get(model)
    if df is None:
        print(f"!! {model}: no master data -> skipped")
        continue
    dfp = df[df['partyid_1'].isin([1, 2])].copy()   # for the Party Identity block
    sheet = pd.read_excel(FEATURE_LIST_FILE, sheet_name=MODEL_SHEET[model])
    sheet = sheet[['attitude', 'category', 'var', 'label', 'measure', 'type']]
    sheet['measure'] = [MEASURE_OVERRIDE.get(v, m) for v, m in zip(sheet['var'], sheet['measure'])]
    sheet = sheet[sheet['var'].apply(lambda x: isinstance(x, str))].copy()
    sheet['catdisp'] = sheet['category'].apply(lambda c: 'Other' if str(c) in ('0', '0.0', 'nan') else str(c))
    # Category in the Excel table: 'Attitude' for attitude variables, otherwise the original category (Demography / SES / Social Capital / Lifestyle)
    sheet['xlsx_cat'] = np.where(sheet['attitude'] == 1, 'Attitude', sheet['catdisp'])
    order = {c: i for i, c in enumerate(CATEGORY_ORDER)}
    sheet['ord'] = sheet['catdisp'].map(lambda c: order.get(c, 999))
    sheet = sheet.sort_values('ord', kind='stable')

    TABLE_RECORDS.clear()
    body = party_block(dfp) + [r'\addlinespace', '']
    n_var, missing = 0, []
    for _, row in sheet.iterrows():
        feat = row['var']
        # For each variable, use only the rows of cells where it entered the model (denominator of the missing share)
        frame = build_feature_frame(data_for_analysis, feat, model)
        if frame is None:
            missing.append(feat)
            continue
        frame = frame[frame['partyid_1'].isin([1, 2])].copy()
        rows = (con_rows if str(row['type']) == 'con' else nom_rows)(frame, feat, row['label'], row['measure'], row['xlsx_cat'])
        if not rows:
            missing.append(feat)
            continue
        body += rows + [r'\addlinespace', '']
        n_var += 1
    while body and body[-1] in (r'\addlinespace', ''):
        body.pop()

    n_dem, n_rep = int((dfp['partyid_1'] == 1).sum()), int((dfp['partyid_1'] == 2).sum())
    tex = preamble(model, n_dem, n_rep) + '\n'.join(body) + '\n\n' + r'\end{longtable}' + '\n'
    with open(MODEL_TEX[model], 'w', encoding='utf-8') as f:
        f.write(tex)
    measures = {v: str(m).strip() for v, m in zip(sheet['var'], sheet['measure']) if _valid_measure(m)}
    write_table_xlsx(TABLE_RECORDS, MODEL_XLSX[model], measures)
    print(f"[Done] {model}: '{MODEL_TEX[model]}' + '{MODEL_XLSX[model]}'  {n_var} variables"
          + (f", excluded {missing}" if missing else ""))

print("\n--- Preview of the table (first 40 lines) ---")
with open(MODEL_TEX['XGBoost'], encoding='utf-8') as f:
    print('\n'.join(f.read().splitlines()[:40]))

## Figures 3 and 4: mean responses of Democrats and Republicans by year

In [ ]:
# === Trend plots of mean responses by year ===
import matplotlib.pyplot as plt
import seaborn as sns

# Party colors: blue (Democrats) and red (Republicans) from the seaborn deep palette
DEM_COLOR = sns.color_palette('deep')[0]
REP_COLOR = sns.color_palette('deep')[3]
PLOT_FOLDER = "outputs/4_descriptive/trend_plots"


def load_features_to_plot(excel_file=FEATURE_LIST_FILE, sheet_name="XGBVARS"):
    """Load var/label/type from a variable sheet (default: XGBVARS = the 66 variables of the final XGBoost model)."""
    print(f"--- Loading variables to plot from '{excel_file}' (sheet '{sheet_name}')... ---")
    cols_needed = ['var', 'label', 'type']
    try:
        dff = pd.read_excel(excel_file, sheet_name=sheet_name)
        if not all(c in dff.columns for c in cols_needed):
            print(f"!! Error: 'var/label/type' columns missing."); return pd.DataFrame(columns=cols_needed)
        dff = dff[cols_needed].dropna(subset=['var', 'type'])
        dff['label'] = dff['label'].fillna(dff['var'])
        print(f"--- {len(dff)} variables loaded. ---")
        return dff
    except FileNotFoundError:
        print(f"!! Error: '{excel_file}' not found."); return pd.DataFrame(columns=cols_needed)
    except Exception as e:
        print(f"!! Error while loading Excel: {e}"); return pd.DataFrame(columns=cols_needed)


def get_yearly_stats(df, feature_name):
    """Weighted mean/SE by year x partyid_1 for continuous/binary variables."""
    df_clean = df[['year', 'partyid_1', 'final_weight', feature_name]].dropna().copy()
    df_clean[feature_name] = pd.to_numeric(df_clean[feature_name], errors='coerce')
    df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce')
    df_clean = df_clean.dropna()

    def calc_stats(group, data_col, weight_col):
        try:
            d = sm_weightstats.DescrStatsW(group[data_col], weights=group[weight_col])
            return pd.Series({'mean': d.mean, 'se': d.std_mean})
        except ZeroDivisionError:
            return pd.Series({'mean': np.nan, 'se': np.nan})

    return df_clean.groupby(['year', 'partyid_1']).apply(
        calc_stats, feature_name, 'final_weight', include_groups=False).reset_index()


def pivot_stats_for_plotting(stats_df):
    df_mean = stats_df.pivot(index='year', columns='partyid_1', values='mean')
    df_se = stats_df.pivot(index='year', columns='partyid_1', values='se')
    return pd.merge(
        df_mean.rename(columns={1: 'Mean_Democrat', 2: 'Mean_Republican'}),
        df_se.rename(columns={1: 'SE_Democrat', 2: 'SE_Republican'}),
        left_index=True, right_index=True, how='left')


def plot_trend_with_error(df_pivot, title, ylabel, filename, y_limit=None):
    """Save a trend plot of means (lines) with 95% CIs (shaded)."""
    plt.figure(figsize=(12, 9))
    sns.set_style("white")
    ci_dem = 1.96 * df_pivot['SE_Democrat']
    ci_rep = 1.96 * df_pivot['SE_Republican']
    plt.plot(df_pivot.index, df_pivot['Mean_Democrat'], label='Democrat (Mean)', color=DEM_COLOR, marker='o')
    plt.fill_between(df_pivot.index, df_pivot['Mean_Democrat'] - ci_dem,
                     df_pivot['Mean_Democrat'] + ci_dem, color=DEM_COLOR, alpha=0.15)
    plt.plot(df_pivot.index, df_pivot['Mean_Republican'], label='Republican (Mean)', color=REP_COLOR, marker='o')
    plt.fill_between(df_pivot.index, df_pivot['Mean_Republican'] - ci_rep,
                     df_pivot['Mean_Republican'] + ci_rep, color=REP_COLOR, alpha=0.15)
    ax = plt.gca()
    ax.set_title(title, fontsize=28, pad=20)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_xlabel("Year", fontsize=18)
    if y_limit:
        ax.set_ylim(y_limit)
    all_years = sorted(df_pivot.index.unique())
    step = 2 if len(all_years) > 10 else 1
    ax.set_xticks(all_years[::step])
    sns.despine()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.clf()
    plt.close()


os.makedirs(PLOT_FOLDER, exist_ok=True)
print(f"--- Plots are saved in '{PLOT_FOLDER}'. ---")
features_to_plot = load_features_to_plot(FEATURE_LIST_FILE, "XGBVARS")
# Variables shown in the paper: Figure 3 (attitude variables of the reduced model in Figure 2-F)
# and Figure 4 (SES variables)
PAPER_FIGURE_VARS = ['abany_c', 'conpress_c', 'eqwlth_c', 'helpblk', 'letin1a', 'racdif1_c',
                     'sei10', 'sei10educ', 'sei10inc']
not_found = sorted(set(PAPER_FIGURE_VARS) - set(features_to_plot['var']))
if not_found:
    raise ValueError(f"Variables not found in XGBVARS: {not_found}")
features_to_plot = features_to_plot[features_to_plot['var'].isin(PAPER_FIGURE_VARS)]
data_for_analysis = load_data_from_pickle(PICKLE_DATA_FILE)

if not features_to_plot.empty and data_for_analysis is not None:
    print("\n" + "=" * 60)
    print(f"Creating trend plots for {len(features_to_plot)} variables...")
    print("=" * 60)
    plot_count = 0
    for _, row in features_to_plot.iterrows():
        feature_name, feature_label, feature_type = row['var'], row['label'], row['type']
        safe_base = feature_name.replace('/', '_').replace(' ', '_')
        # For each variable, use only the rows of cells where it entered the model
        df_analysis = build_feature_frame(data_for_analysis, feature_name, 'XGBoost')
        if df_analysis is None:
            print(f"!! [Skip] '{feature_name}': not in data."); continue
        df_analysis = df_analysis[df_analysis['partyid_1'].isin([1, 2])].copy()
        try:
            if feature_type == 'con':
                print(f"--- [con] {feature_name} ({feature_label}) ---")
                is_binary = df_analysis[feature_name].dropna().isin([0, 1]).all()
                y_limit = (0, 1) if is_binary else None
                ylabel = "Proportion (Mean)" if is_binary else "Mean (Weighted)"
                pivot_df = pivot_stats_for_plotting(get_yearly_stats(df_analysis, feature_name))
                if pivot_df.empty:
                    print(f"!! [Skip] '{feature_name}': empty statistics."); continue
                plot_trend_with_error(pivot_df, f"'{feature_label}'", ylabel,
                                      os.path.join(PLOT_FOLDER, f"trend_con_{safe_base}.png"), y_limit)
                plot_count += 1
            elif feature_type == 'nom':
                print(f"--- [nom] {feature_name} ({feature_label}) ---")
                categories = sorted(df_analysis[feature_name].dropna().unique())
                if len(categories) > 10:
                    print(f"!! [Skip] '{feature_name}': {len(categories)} categories (>10)."); continue
                # Denominator = all respondents including missing values (in the cells used) -> same as the category % of the table
                for cat in categories:
                    cat_str = str(cat).replace('.0', '')
                    dummy_name = f"{feature_name}_dummy_{cat_str}"
                    df_analysis[dummy_name] = np.where(df_analysis[feature_name] == cat, 1, 0)
                    pivot_df = pivot_stats_for_plotting(get_yearly_stats(df_analysis, dummy_name))
                    if pivot_df.empty:
                        print(f"!! [Skip] '{feature_name}' (Cat {cat_str}): empty."); continue
                    plot_trend_with_error(pivot_df, f"'{feature_label} (Cat: {cat_str})'", "Proportion (Mean)",
                                          os.path.join(PLOT_FOLDER, f"trend_nom_{safe_base}_cat_{cat_str}.png"), y_limit=(0, 1))
                    plot_count += 1
            else:
                print(f"!! [Skip] '{feature_name}': type='{feature_type}'.")
        except Exception as e:
            print(f"!! [ERROR] '{feature_name}': {e}")
    print("\n" + "=" * 60)
    print(f"[Done] {plot_count} plots saved in '{PLOT_FOLDER}'.")
    print("=" * 60)
else:
    print("!! Failed to load the variable list or the data.")